# Lab 18: Spark MLlib

**Module 18** | Read `notes/18-spark-mllib.pdf` before running this notebook.

Build a Spark ML pipeline on the credit-fraud sample and compare fit time with scikit-learn on the same rows.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Distributed ML with PySpark MLlib

Credit-card fraud detection is a classic imbalanced classification problem.
The sample dataset contains anonymized features **V1 to V28** (PCA components) plus `Amount`, `Time`, and a binary `Class` label.

Spark MLlib runs pipelines across a cluster (here: `local[1]` for a single machine) using DataFrame operations.
We build a **Pipeline** with `VectorAssembler` → `RandomForestClassifier`, time the fit, and compare against **scikit-learn** on the same rows.

If `credit_fraud_sample.parquet` is missing (pyarrow not installed), the lab falls back to the CSV version automatically.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime, create_spark, stop_spark

configure_runtime(quiet=True)

import time

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

parquet_path = ROOT / "datasets" / "credit_fraud_sample.parquet"
csv_path = ROOT / "datasets" / "credit_fraud_sample.csv"

spark = create_spark("Lab18")
if parquet_path.exists():
    df = spark.read.parquet(str(parquet_path))
    print(f"Loaded parquet: {parquet_path.name} ({df.count()} rows)")
else:
    df = spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))
    print(f"Loaded CSV: {csv_path.name} ({df.count()} rows)")

feature_cols = [f"V{i}" for i in range(1, 29)]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
rf = RandomForestClassifier(
    labelCol="Class",
    featuresCol="features",
    numTrees=50,
    maxDepth=8,
    seed=42,
)
pipeline = Pipeline(stages=[assembler, rf])

train, test = df.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train.count()}, test rows: {test.count()}")

t0 = time.perf_counter()
spark_model = pipeline.fit(train)
spark_fit_sec = time.perf_counter() - t0

predictions = spark_model.transform(test)
evaluator = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction", metricName="accuracy"
)
spark_accuracy = evaluator.evaluate(predictions)
print(f"Spark fit time: {spark_fit_sec:.3f} s")
print(f"Spark test accuracy: {spark_accuracy:.4f}")

stop_spark(spark)


In [ ]:
import time

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

if parquet_path.exists():
    pdf = pd.read_parquet(parquet_path)
else:
    pdf = pd.read_csv(csv_path)

X = pdf[feature_cols]
y = pdf["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

t0 = time.perf_counter()
sk_model = RandomForestClassifier(
    n_estimators=50, max_depth=8, random_state=42, n_jobs=-1
)
sk_model.fit(X_train, y_train)
sklearn_fit_sec = time.perf_counter() - t0

sk_accuracy = accuracy_score(y_test, sk_model.predict(X_test))
print(f"sklearn fit time: {sklearn_fit_sec:.3f} s")
print(f"sklearn test accuracy: {sk_accuracy:.4f}")
print(f"Fit-time ratio Spark/sklearn: {spark_fit_sec / sklearn_fit_sec:.2f}x")


## Spark vs scikit-learn

On a **small local sample** (~5k rows), scikit-learn usually **fits faster** because Spark pays JVM startup, DataFrame planning, and serialization overhead.
Spark's advantage appears at **millions+ of rows** on a cluster where data does not fit in memory on one machine.

Accuracies should be **close but not identical**, different random splits (Spark `randomSplit` vs sklearn `train_test_split`) and implementation details cause small differences.

**When to use which:**

- **sklearn:** prototyping, datasets that fit in RAM, single-machine speed.
- **Spark MLlib:** large-scale ETL + training in one pipeline, data already in a lake/warehouse, distributed feature engineering.
